Assignment 3: Visualization in Python

In [1]:
# Import libraries
import pandas as pd
import plotly.graph_objects as go

# Read the CSV file
df = pd.read_csv('2024-25_cepg_recipients_web_en.csv')
print(df.columns.to_list())

# Create a function to parse and categorize expenditures based on the "Description" column
def categorize_recipient(Recipient):
    recipient_lower = Recipient.lower()
    if 'first nation' in recipient_lower or 'band council' in recipient_lower or 'mohawk' in recipient_lower:
        return 'First Nations'
    elif 'corporation of the city' in recipient_lower or 'city of' in recipient_lower:
        return 'Cities'
    elif 'corporation of the county' in recipient_lower or 'county of' in recipient_lower:
        return 'Counties'
    elif 'corporation of the town' in recipient_lower or 'town of' in recipient_lower:
        return 'Towns'
    elif 'corporation of the township' in recipient_lower or 'township of' in recipient_lower or 'corporation of the municipality' in recipient_lower or 'municipality of' in recipient_lower:
        return 'Townships/Municipalities'
    elif 'local services board' in recipient_lower:
        return 'Local Services Boards'
    elif 'search and rescue' in recipient_lower or 'fire protection' in recipient_lower:
        return 'Emergency Services'
    else:
        return 'Other Organizations'
    
# Create a function to categorize recipients
def categorize_expenditure(Description):
    description_lower = Description.lower()
    if 'generator' in description_lower:
        return 'Generators & Power'
    elif 'drone' in description_lower:
        return 'Drones & Surveillance'
    elif 'radio' in description_lower or 'communication' in description_lower:
        return 'Communications Equipment'
    elif 'flood' in description_lower or 'sandbag' in description_lower or 'pump' in description_lower:
        return 'Flood Response'
    elif 'fire' in description_lower or 'wildland' in description_lower or 'chainsaw' in description_lower:
        return 'Fire Response'
    elif 'training' in description_lower or 'education' in description_lower:
        return 'Training & Education'
    elif 'shelter' in description_lower or 'evacuation' in description_lower or 'emergency centre' in description_lower:
        return 'Emergency Shelters'
    elif 'rescue' in description_lower:
        return 'Search & Rescue'
    else:
        return 'Other Equipment'
    
# Apply categorization functions defined above
df['Recipient Type'] = df['Recipient'].apply(categorize_recipient)
df['Expenditure Category'] = df['Description'].apply(categorize_expenditure)

# Create funding distribution by recipient type and equipment category
recipient_funding = df.groupby('Recipient Type')['Funding $'].sum().reset_index()
expenditure_funding = df.groupby('Expenditure Category')['Funding $'].sum().reset_index()

# For demonstration, let's create a simple Sankey showing flow from recipient types to equipment categories
flow_data = df.groupby(['Recipient Type', 'Expenditure Category'])['Funding $'].sum().reset_index()

# Prepare data for Sankey diagram
# Create unique labels for source and target
source_labels = list(recipient_funding['Recipient Type'].unique())
target_labels = list(expenditure_funding['Expenditure Category'].unique())
all_labels = source_labels + target_labels

# Create mappings
source_indices = {label: i for i, label in enumerate(source_labels)}
target_indices = {label: i + len(source_labels) for i, label in enumerate(target_labels)}

# Prepare source, target, and value arrays
source = []
target = []
values = []

for _, row in flow_data.iterrows():
    source.append(source_indices[row['Recipient Type']])
    target.append(target_indices[row['Expenditure Category']])
    values.append(row['Funding $'])

# Create the Sankey diagram!
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_labels,
        color=["lightblue"] * len(source_labels) + ["lightcoral"] * len(target_labels)
    ),
    link=dict(
        source=source,
        target=target,
        value=values,
        color="rgba(0,0,255,0.3)"
    )
)])

fig.update_layout(
    title_text="Community Emergency Preparedness Grant Funding Flow<br>"+
                "Source: Open Ontario Data Catalogue (May 2025)",
    font_size=12,
    height=600,
    width=1000
)

# Display the plot
fig.show()

# Save visualization
fig.write_html("sankey_cepg_funding.html")

['Recipient', 'Funding $', 'Description']
